# Run ESS-LLM Simulation (modular `ess_sim/` engine)

This notebook is the **run entry point** for the `ess_sim/` package (the logic
extracted from `ess_simulation.ipynb`). Edit/debug the engine in the `.py`
modules; run and inspect here. The original `ess_simulation.ipynb` is untouched.


In [ ]:
# Imports -- pull the engine from the ess_sim package
import os, json, random, time
import numpy as np
import pandas as pd

from ess_sim.config import ExperimentConfig, generate_ess_conditions
from ess_sim.model import World
from ess_sim.article import fetch_and_cache
from ess_sim.ess_pool import _load_ess_pool, _ESS_POOL_PATH, run_pool_diagnostics
from ess_sim.network import (generate_network, load_network_structure,
                             save_network_structure, get_network_statistics)
from ess_sim.prompt import build_ess_persona_prompt
from ess_sim.llm_client import LLM_STATS
from ess_sim import visualization as viz
print('ess_sim engine imported.')

In [ ]:
# B0 -- API key check (project uses Azure OpenAI only; see ess_sim/llm_client.py)
print("=" * 50)
for name in ("AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY"):
    val = os.environ.get(name, "")
    print(f"  {name:<24} {'SET' if val else 'NOT SET'}")
print("=" * 50)
print("Both Azure vars must be SET before running simulations.")

In [ ]:
# B1 -- Experiment parameters
# --- Single-condition parameters (the B2-B12 walk-through of ONE example run) ---
NETWORK_TYPE = "small_world"     # centralized | small_world | random
NUM_AGENTS   = 30
STEP_COUNT   = 20
SEED         = 42
INTERVENTION = "none"            # none | fca | denial  (3-level intervention)

# --- Matrix run (B11): conditions come from generate_ess_conditions(); counts printed, not hardcoded ---
# RUN_MODE:
#   "single"  -> single condition above only (B2-B12); matrix skipped
#   "medium"  -> generate_ess_conditions(seeds=[42])         (1 seed  x 3 networks x 3 interventions: none/fca/denial)
#   "full"    -> generate_ess_conditions(seeds=MATRIX_SEEDS) (3 seeds x 3 networks x 3 interventions: none/fca/denial)
RUN_MODE     = "medium"
MATRIX_SEEDS = [42, 123, 777]

cfg = ExperimentConfig(
    network_type     = NETWORK_TYPE,
    num_agents       = NUM_AGENTS,
    step_count       = STEP_COUNT,
    seed             = SEED,
    fca_enabled      = (INTERVENTION == "fca"),
    denial_enabled   = (INTERVENTION == "denial"),
    max_interactions = 1,
)
print(f"Exp ID      : {cfg.exp_id}")
print(f"Network     : {cfg.network_type} | N={cfg.num_agents} | Steps={cfg.step_count}")
print(f"Contact     : {cfg.max_interactions} random neighbour per step (Deffuant convention)")
print(f"Intervention: {INTERVENTION} | Model: {cfg.gpt_model}")
if RUN_MODE == "single":
    print("RUN_MODE    : single (matrix skipped)")
else:
    _seeds = [42] if RUN_MODE == "medium" else MATRIX_SEEDS
    _n = len(generate_ess_conditions(num_agents=NUM_AGENTS, step_count=STEP_COUNT, seeds=_seeds))
    print(f"RUN_MODE    : {RUN_MODE} | {_n} conditions from generate_ess_conditions (seeds={_seeds})")

In [ ]:
# B2 -- Article fetch + ESS respondent pool check (pool now read from CSV)
topic_data = fetch_and_cache("climate_europe", cache_dir="./data")
print(f"Question : {topic_data['question']}")
print(f"Article  : {len(topic_data['article'])} chars")
print(f"Preview  : {topic_data['article'][:300]}...")

print()
if os.path.exists(_ESS_POOL_PATH):
    _pool_size = len(_load_ess_pool())
    print(f"[INFO] ESS respondent pool found: {_pool_size} records ({_ESS_POOL_PATH})")
else:
    print(f"[ERROR] ESS respondent pool NOT found at {_ESS_POOL_PATH}")
    print("  -> Export data/ess_respondent_pool.csv from the JSON first.")

In [ ]:
# B3 -- Network preview (identical generate_network + cache-file naming as World.__init__ -> drift-safe).
# No global random.seed needed: generate_network is reproducible via its own seed= argument.
network_file = (
    f"./data/{cfg.network_type}_network_"
    f"num_agents_{cfg.num_agents}_seed_{cfg.network_seed}.json"
)
if os.path.exists(network_file):
    G_preview = load_network_structure(network_file)
    print(f"[CACHE] Loaded network from {network_file}")
else:
    G_preview = generate_network(cfg.network_type, cfg.num_agents, seed=cfg.network_seed)
    save_network_structure(G_preview, network_file)
    print(f"[NEW] Generated {cfg.network_type} network")

stats = get_network_statistics(G_preview)
for k, v in stats.items():
    print(f"  {k:<30} {v}")

In [ ]:
# B4 -- Network topology comparison (ess_sim.visualization)
nets = viz.plot_topology_comparison(cfg.num_agents, cfg.network_seed,
                                    highlight_fca=True, show_degree_dist=True,
                                    save="experiments_ess/network_comparison.png")

In [ ]:
# B4b -- ESS pool diagnostics
# Who the agents are drawn from. Prints tables A.1 (concern distribution, with the
# Sarle coefficient confirming the pool is unimodal) and B (sample characteristics),
# and saves both to data/ for the paper. Build the pool first if it is missing:
# `python -m ess_sim.ess_pool`
run_pool_diagnostics()


In [ ]:
# B5 -- ESS Persona preview (sample 4 respondents)
rng = random.Random(cfg.seed + 999)
pool = _load_ess_pool()
sample_rows = rng.sample(pool, min(4, len(pool)))
for row in sample_rows:
    prompt, profile = build_ess_persona_prompt(row)
    print(f"\n{'='*60}")
    print(f"Profile : {profile}")
    print(f"Prompt  : {prompt[:300]}...")

## Run simulation (below cells make LLM calls -- cost money)

In [ ]:
# B6 -- Create World (LLM initialization)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

world = World(cfg, load_network=True)


In [ ]:
# B8 -- Run simulation
world.run_model(step_count=cfg.step_count)

output_dir = os.path.join(cfg.exp_dir, cfg.exp_id)
print(f"\n[DONE] Output dir: {output_dir}")
for f in sorted(os.listdir(output_dir)):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        print(f"  {f:<45} {os.path.getsize(path):>8} bytes")

In [ ]:
# B10b -- Interaction diary (load the just-saved agents_data.json)
agents_data = json.load(open(os.path.join(cfg.exp_dir, cfg.exp_id, 'agents_data.json'), encoding='utf-8'))
viz.interaction_diary(agents_data, max_steps=2)

## Experiment matrix (B11) and cross-condition summary (B12)

In [ ]:
# B11 -- Experiment matrix
# The runner lives in run_full.py (single source of truth: resumability, seeding,
# cost accounting). Do not re-implement it here -- the two copies used to drift.
from run_full import run as run_matrix

if RUN_MODE == "medium":
    configs = generate_ess_conditions(num_agents=NUM_AGENTS, step_count=STEP_COUNT, seeds=[42])
    print('Medium run: {} conditions  (seed=42, N={}, {} steps)'.format(len(configs), NUM_AGENTS, STEP_COUNT))
    run_matrix(configs)
elif RUN_MODE == "full":
    configs = generate_ess_conditions(num_agents=NUM_AGENTS, step_count=STEP_COUNT, seeds=MATRIX_SEEDS)
    print('Full run: {} conditions  (seeds={}, N={}, {} steps)'.format(len(configs), MATRIX_SEEDS, NUM_AGENTS, STEP_COUNT))
    run_matrix(configs)
else:
    print("RUN_MODE='{}' -> skipped. Set to 'medium' or 'full' in B1 to run.".format(RUN_MODE))
    print('Preview (full conditions):')
    for c in generate_ess_conditions(num_agents=NUM_AGENTS, step_count=STEP_COUNT, seeds=MATRIX_SEEDS):
        print('  {}'.format(c.exp_id))


In [ ]:
# B12 -- Cross-condition summary table (final step, by condition)
results = []
for run_cfg in generate_ess_conditions(num_agents=NUM_AGENTS, step_count=STEP_COUNT, seeds=MATRIX_SEEDS):
    overview_path = os.path.join(run_cfg.exp_dir, run_cfg.exp_id, "model_overview.json")
    if not os.path.exists(overview_path):
        continue
    with open(overview_path, "r", encoding="utf-8") as f:
        lines = [json.loads(l) for l in f if l.strip()]
    final = lines[-1] if lines else {}
    results.append({
        "network":      run_cfg.network_type,
        # 3-level, not a boolean: fca_enabled=False is BOTH 'none' and 'denial'.
        "intervention": "fca" if run_cfg.fca_enabled else ("denial" if run_cfg.denial_enabled else "none"),
        "seed":         run_cfg.seed,
        "MeanConcern":  final.get("mean_concern", float("nan")),
        "Dispersion":   final.get("dispersion", float("nan")),
        "Bimodality":   final.get("bimodality", float("nan")),
        "CIR":          final.get("cross_camp_rate", float("nan")),
        "llm_failures": final.get("llm_failures", 0),
    })

if results:
    df = pd.DataFrame(results)
    print("=== Per-condition results (final step) ===")
    print(df.to_string(index=False))
    print("\nMean concern: {:.4f} | Mean Disp: {:.4f}".format(df["MeanConcern"].mean(), df["Dispersion"].mean()))
    if df["llm_failures"].sum():
        print("[WARN] {} LLM failure(s) across these runs were recorded as 'no change'.".format(
            int(df["llm_failures"].sum())))
else:
    print("No completed runs found. Run B11 first (set RUN_MODE='full').")
